In [ ]:
import os, random
import numpy as np
import pandas as pd
from pathlib import Path

import soundfile as sf
import librosa

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
XGB_AVAILABLE = True

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass
from tqdm import tqdm

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix
)

from scipy.stats import randint, uniform

In [ ]:
!pip install -q wandb
import wandb
wandb.login()
WANDB_PROJECT = "speech-defects-experiments"
WANDB_GROUP = "baselines"
WANDB_ENABLED = True

def start_baseline_run(name, config):
    if not WANDB_ENABLED:
        return None
    run = wandb.init(
        project=WANDB_PROJECT,
        group=WANDB_GROUP,
        name=name,
        config=config,
        reinit=True,
    )
    return run

def finish_baseline_run(run):
    if run is not None:
        run.finish()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: aksenovmr (aksenovmr-hse-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [ ]:
def metrics_dict(y_true, y_pred, y_score_bad, thr):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    return {
        "thr": float(thr),
        "auc": float(roc_auc_score(y_true, y_score_bad)),
        "ap": float(average_precision_score(y_true, y_score_bad)),
        "acc": float(accuracy_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred, average="binary", pos_label=1)),
        "f1_macro": float(f1_score(y_true, y_pred, average="macro")),
        "precision": float(precision_score(y_true, y_pred, average="binary", pos_label=1, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, average="binary", pos_label=1, zero_division=0)),
    }

def log_baseline_metrics(
    run,
    val_default,
    val_best,
    test_default,
    test_best,
    thr_bad,
    extra=None,
):
    if run is None:
        return

    payload = {
        "thresholds/thr_bad": float(thr_bad),

        "val/default_auc": val_default["auc"],
        "val/default_ap": val_default["ap"],
        "val/default_acc": val_default["acc"],
        "val/default_f1": val_default["f1"],
        "val/default_f1_macro": val_default["f1_macro"],
        "val/default_precision": val_default["precision"],
        "val/default_recall": val_default["recall"],

        "val/best_f1_auc": val_best["auc"],
        "val/best_f1_ap": val_best["ap"],
        "val/best_f1_acc": val_best["acc"],
        "val/best_f1_f1": val_best["f1"],
        "val/best_f1_f1_macro": val_best["f1_macro"],
        "val/best_f1_precision": val_best["precision"],
        "val/best_f1_recall": val_best["recall"],

        "test/default_auc": test_default["auc"],
        "test/default_ap": test_default["ap"],
        "test/default_acc": test_default["acc"],
        "test/default_f1": test_default["f1"],
        "test/default_f1_macro": test_default["f1_macro"],
        "test/default_precision": test_default["precision"],
        "test/default_recall": test_default["recall"],

        "test/best_f1_from_val_auc": test_best["auc"],
        "test/best_f1_from_val_ap": test_best["ap"],
        "test/best_f1_from_val_acc": test_best["acc"],
        "test/best_f1_from_val_f1": test_best["f1"],
        "test/best_f1_from_val_f1_macro": test_best["f1_macro"],
        "test/best_f1_from_val_precision": test_best["precision"],
        "test/best_f1_from_val_recall": test_best["recall"],
    }

    if extra:
        payload.update(extra)

    wandb.log(payload)

In [ ]:
GOOD_ZIP = "/content/хорошо-20260103T074218Z-1-001.zip"
BAD_ZIP  = "/content/плохо-20260103T074214Z-1-001.zip"

GOOD_DIR = Path("/content/Хорошо")
BAD_DIR  = Path("/content/Плохо")
GOOD_DIR.mkdir(parents=True, exist_ok=True)
BAD_DIR.mkdir(parents=True, exist_ok=True)

!ls -lh "/content"
!file "{GOOD_ZIP}"
!file "{BAD_ZIP}"

!apt-get -qq update
!apt-get -qq install -y p7zip-full

!7z x "{GOOD_ZIP}" -o"{GOOD_DIR}" -y
!7z x "{BAD_ZIP}"  -o"{BAD_DIR}"  -y

good_wavs = sorted(GOOD_DIR.rglob("*.wav"))
bad_wavs  = sorted(BAD_DIR.rglob("*.wav"))

total 666M
drwxr-xr-x 1 root root 4.0K Apr 16 13:33 sample_data
drwxr-xr-x 2 root root 4.0K Apr 26 14:30 Плохо
-rw-r--r-- 1 root root 274M Apr 26 14:27 плохо-20260103T074214Z-1-001.zip
drwxr-xr-x 2 root root 4.0K Apr 26 14:30 Хорошо
-rw-r--r-- 1 root root 392M Apr 26 14:28 хорошо-20260103T074218Z-1-001.zip
/content/хорошо-20260103T074218Z-1-001.zip: Zip archive data, at least v2.0 to extract, compression method=deflate
/content/плохо-20260103T074214Z-1-001.zip: Zip archive data, at least v2.0 to extract, compression method=deflate
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)

7-Zip [64] 16.02 : Copyright (c) 1999-2016 Igor Pavlov : 2016-05-21
p7zip Version 16.02 (locale=en_US.UTF-8,Utf16=on,HugeFiles=on,64 bits,2 CPUs Intel(R) Xeon(R) CPU @ 2.00GHz (50653),ASM,AES-NI)

Scanning the drive for archives:
  0M Scan /content/     

In [ ]:
seed = 42
rows=[]
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(seed)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

device: cuda


In [ ]:
df = pd.DataFrame({
    "path": [str(p) for p in good_wavs] + [str(p) for p in bad_wavs],
    "label": [0]*len(good_wavs) + [1]*len(bad_wavs)})
print("\ndf shape:", df.shape)
df.sample(5, random_state=seed)


df shape: (2792, 2)


,path,label
2653,/content/Плохо/плохо/eu.cc3c6be0-524b-4da9-818...,1
1192,/content/Хорошо/хорошо/eu.a86e47ac-920a-4b6b-a...,0
2683,/content/Плохо/плохо/eu.d9c40738-6d48-44fb-804...,1
141,/content/Хорошо/хорошо/eu.14c51714-2ec6-4ba3-9...,0
211,/content/Хорошо/хорошо/eu.1ef271fb-ca8e-496d-b...,0


In [ ]:
def get_duration_sec(path, target_sr=None):
    x, sr = librosa.load(path, sr=target_sr, mono=True)
    return len(x) / sr

df["duration_sec"] = df["path"].apply(lambda p: get_duration_sec(p, target_sr=None))
df.groupby("label")["duration_sec"].describe()

,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
0,1885.0,8.896937,4.487628,2.28,5.98,7.86,10.64,44.10
1,907.0,13.345126,7.384369,0.00,8.48,11.76,16.24,68.66


In [ ]:
bad_rows = df[df['duration_sec']==0]

df.drop(bad_rows.index, inplace=True)

In [ ]:
def get_score_for_auc(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        score = model.decision_function(X)
        score = (score - score.min()) / (score.max() - score.min() + 1e-8)
        return score
    return None

def add_row(rows, name, y_true, y_pred_bad, y_score_bad=None):
    row = {
        "Model": name,
        "Accuracy": accuracy_score(y_true, y_pred_bad),
        "F1_bad": f1_score(y_true, y_pred_bad, average="binary", pos_label=1),
        "F1_macro": f1_score(y_true, y_pred_bad, average="macro"),
        "Precision_bad": precision_score(y_true, y_pred_bad, average="binary", pos_label=1, zero_division=0),
        "Recall_bad": recall_score(y_true, y_pred_bad, average="binary", pos_label=1, zero_division=0),
        "Recall_good": recall_score(y_true, y_pred_bad, average="binary", pos_label=0, zero_division=0),
    }
    if y_score_bad is not None:
        row["ROC_AUC"] = roc_auc_score(y_true, y_score_bad)
        row["AP"] = average_precision_score(y_true, y_score_bad)
    rows.append(row)
    return rows

def print_report(name, y_true, y_pred_bad, y_score_bad):
    acc = accuracy_score(y_true, y_pred_bad)
    f1_bad = f1_score(y_true, y_pred_bad, average="binary", pos_label=1)
    f1_mac = f1_score(y_true, y_pred_bad, average="macro")
    prec_bad = precision_score(y_true, y_pred_bad, average="binary", pos_label=1, zero_division=0)
    rec_bad = recall_score(y_true, y_pred_bad, average="binary", pos_label=1, zero_division=0)
    auc = roc_auc_score(y_true, y_score_bad)
    ap = average_precision_score(y_true, y_score_bad)

    print(f"[{name}]")
    print(f"  Accuracy      : {acc:.4f}")
    print(f"  F1-bad        : {f1_bad:.4f}")
    print(f"  F1-macro      : {f1_mac:.4f}")
    print(f"  Precision_bad : {prec_bad:.4f}")
    print(f"  Recall_bad    : {rec_bad:.4f}")
    print(f"  ROC-AUC       : {auc:.4f}")
    print(f"  AP            : {ap:.4f}")

In [ ]:
df_train_tab, df_test_tab = train_test_split(
    df, test_size=0.2, random_state=seed, stratify=df["label"].values
)

df_train_base, df_val_base = train_test_split(
    df_train_tab,
    test_size=0.2,
    random_state=seed,
    stratify=df_train_tab["label"].values
)

df_train_base = df_train_base.reset_index(drop=True)
df_val_base   = df_val_base.reset_index(drop=True)
df_test_base  = df_test_tab.reset_index(drop=True)

print("Train:", len(df_train_base), "Val:", len(df_val_base), "Test:", len(df_test_base))
print("Train counts:\n", df_train_base["label"].value_counts())
print("Val counts:\n", df_val_base["label"].value_counts())
print("Test counts:\n", df_test_base["label"].value_counts())

Train: 1785 Val: 447 Test: 558
Train counts:
 label
0    1206
1     579
Name: count, dtype: int64
Val counts:
 label
0    302
1    145
Name: count, dtype: int64
Test counts:
 label
0    377
1    181
Name: count, dtype: int64


In [ ]:
df_test_base

,path,label,duration_sec
0,/content/Хорошо/хорошо/eu.6fbeb16c-7205-41bd-b...,0,7.476438
1,/content/Хорошо/хорошо/eu.170b6ac4-c85c-4b49-a...,0,6.919125
2,/content/Хорошо/хорошо/eu.042fbe10-7a1e-45e8-b...,0,6.700000
3,/content/Плохо/плохо/eu.ee7fda9a-684c-4bec-aad...,1,16.340000
4,/content/Плохо/плохо/eu.e7e89c69-2e96-4368-bf7...,1,15.180000
...,...,...,...
553,/content/Хорошо/хорошо/eu.425981eb-3f70-47f4-a...,0,5.840000
554,/content/Хорошо/хорошо/eu.be9afb05-4047-44ff-a...,0,5.160000
555,/content/Хорошо/хорошо/eu.038f9ff1-d921-46e2-b...,0,7.660000
556,/content/Хорошо/хорошо/eu.ab596c07-6d15-4e8f-9...,0,13.060000


## Модели на MFCC-признаках

In [ ]:
SR = 16000
MAX_SECONDS = 10.0
MAX_LEN = int(SR * MAX_SECONDS)

def load_audio(path, sr=SR):
    errors = []
    try:
        x, file_sr = sf.read(path, always_2d=False)
        if x.ndim > 1:
            x = np.mean(x, axis=1)
        x = x.astype(np.float32)
        if file_sr != sr:
            x = librosa.resample(x, orig_sr=file_sr, target_sr=sr)
        return x.astype(np.float32)
    except Exception as e:
        errors.append(f"soundfile: {repr(e)}")

    try:
        x, _ = librosa.load(path, sr=sr, mono=True)
        return x.astype(np.float32)
    except Exception as e:
        errors.append(f"librosa: {repr(e)}")

    raise RuntimeError("Не удалось прочитать аудиофайл: " + " | ".join(errors))

def pad_or_crop(x, max_len=MAX_LEN, train=False):
    if len(x) == 0:
        return np.zeros((max_len,), dtype=np.float32)

    if len(x) > max_len:
        if train:
            start = np.random.randint(0, len(x) - max_len + 1)
        else:
            start = (len(x) - max_len) // 2
        return x[start:start + max_len]

    if len(x) < max_len:
        return np.pad(x, (0, max_len - len(x)), mode="constant")

    return x

In [ ]:
def extract_mfcc_features(path, sr=SR, n_mfcc=13, n_fft=1024, hop_length=256, train=False):
    x = load_audio(path, sr=sr)
    x = pad_or_crop(x, max_len=MAX_LEN, train=train)

    mfcc = librosa.feature.mfcc(
        y=x, sr=sr, n_mfcc=n_mfcc, n_fft=n_fft, hop_length=hop_length
    )
    feat = np.concatenate([mfcc.mean(axis=1), mfcc.std(axis=1)], axis=0)
    return feat.astype(np.float32)

def build_Xy_from_df(df_part, n_mfcc=13, train=False):
    X = np.stack([
        extract_mfcc_features(p, n_mfcc=n_mfcc, train=train)
        for p in tqdm(df_part["path"].tolist())
    ])
    y = df_part["label"].astype(int).values
    return X, y

In [ ]:
def tune_threshold_by_f1(y_true, y_score_bad):
    thresholds = np.linspace(0.01, 0.99, 99)
    best_thr = 0.5
    best_f1 = -1.0

    y_true = np.asarray(y_true)

    for thr in thresholds:
        y_pred = (y_score_bad >= thr).astype(int)
        f1 = f1_score(y_true, y_pred, average="binary", pos_label=1)
        if f1 > best_f1:
            best_f1 = f1
            best_thr = thr

    return best_thr, best_f1

In [ ]:
N_MFCC = 20

X_train, y_train = build_Xy_from_df(df_train_base, n_mfcc=N_MFCC, train=True)
X_val,   y_val   = build_Xy_from_df(df_val_base,   n_mfcc=N_MFCC, train=False)
X_test,  y_test  = build_Xy_from_df(df_test_base,  n_mfcc=N_MFCC, train=False)

print("X_train:", X_train.shape, "X_val:", X_val.shape, "X_test:", X_test.shape)

100%|██████████| 558/558 [00:07<00:00, 74.69it/s]

X_train: (1785, 40) X_val: (447, 40) X_test: (558, 40)


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)

### Логистическая регрессия

In [ ]:
run = start_baseline_run(
    name="mfcc_logreg",
    config={
        "model_family": "baseline",
        "features": "mfcc",
        "classifier": "logreg",
        "sr": SR,
        "max_seconds": MAX_SECONDS,
        "random_state": seed,
        "n_mfcc": N_MFCC,
    },
)

pipe_lr = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=5000))
])

param_grid_lr = [
    {
        "clf__penalty": ["l2"],
        "clf__solver": ["lbfgs", "liblinear"],
        "clf__C": [0.01, 0.1, 1, 3, 10, 30],
        "clf__class_weight": [None, "balanced"],
    },
    {
        "clf__penalty": ["l1"],
        "clf__solver": ["liblinear"],
        "clf__C": [0.01, 0.1, 1, 3, 10, 30],
        "clf__class_weight": [None, "balanced"],
    },
    {
        "clf__penalty": ["elasticnet"],
        "clf__solver": ["saga"],
        "clf__C": [0.01, 0.1, 1, 3, 10],
        "clf__l1_ratio": [0.1, 0.3, 0.5, 0.7, 0.9],
        "clf__class_weight": [None, "balanced"],
    },
]

gs_lr = GridSearchCV(
    estimator=pipe_lr,
    param_grid=param_grid_lr,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
    verbose=1
)
gs_lr.fit(X_train, y_train)
print("\n[LR] Best CV macro-F1:", gs_lr.best_score_)
print("[LR] Best params:", gs_lr.best_params_)
logreg_tuned = gs_lr.best_estimator_

val_score_bad = get_score_for_auc(logreg_tuned, X_val)
best_thr, best_val_f1 = tune_threshold_by_f1(y_val, val_score_bad)

test_score_bad = get_score_for_auc(logreg_tuned, X_test)

val_pred_default = (val_score_bad >= 0.5).astype(int)
test_pred_default = (test_score_bad >= 0.5).astype(int)

val_pred_best = (val_score_bad >= best_thr).astype(int)
test_pred_best = (test_score_bad >= best_thr).astype(int)

val_metrics_default = metrics_dict(y_val, val_pred_default, val_score_bad, 0.5)
val_metrics_best = metrics_dict(y_val, val_pred_best, val_score_bad, best_thr)

test_metrics_default = metrics_dict(y_test, test_pred_default, test_score_bad, 0.5)
test_metrics_best = metrics_dict(y_test, test_pred_best, test_score_bad, best_thr)

print("Best threshold (LogReg, bad):", best_thr, "| Val F1_bad:", best_val_f1)
print_report("LogReg tuned (MFCC)", y_test, test_pred_best, test_score_bad)
add_row(rows, "LogReg tuned (MFCC)", y_test, test_pred_best, test_score_bad)

log_baseline_metrics(
    run=run,
    val_default=val_metrics_default,
    val_best=val_metrics_best,
    test_default=test_metrics_default,
    test_best=test_metrics_best,
    thr_bad=best_thr,
    extra={
        "cv/best_score": float(gs_lr.best_score_),
        "model/name": "mfcc_logreg",
    },
)

finish_baseline_run(run)

Fitting 5 folds for each of 86 candidates, totalling 430 fits

[LR] Best CV macro-F1: 0.6885174022848133
[LR] Best params: {'clf__C': 0.1, 'clf__class_weight': None, 'clf__penalty': 'l2', 'clf__solver': 'liblinear'}
Best threshold (LogReg, bad): 0.36000000000000004 | Val F1_bad: 0.5531914893617021
[LogReg tuned (MFCC)]
  Accuracy      : 0.6505
  F1-bad        : 0.5161
  F1-macro      : 0.6213
  Precision_bad : 0.4685
  Recall_bad    : 0.5746
  ROC-AUC       : 0.6720
  AP            : 0.5113


cv/best_score,▁
test/best_f1_from_val_acc,▁
test/best_f1_from_val_ap,▁
test/best_f1_from_val_auc,▁
test/best_f1_from_val_f1,▁
test/best_f1_from_val_f1_macro,▁
test/best_f1_from_val_precision,▁
test/best_f1_from_val_recall,▁
test/default_acc,▁
test/default_ap,▁
+20,...


### SVM

In [ ]:
run = start_baseline_run(
    name="mfcc_svm",
    config={
        "model_family": "baseline",
        "features": "mfcc",
        "classifier": "svm",
        "sr": SR,
        "max_seconds": MAX_SECONDS,
        "random_state": seed,
        "n_mfcc": N_MFCC,
    },
)

pipe_svm = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", SVC(probability=True))
])

param_grid_svm = [
    {
        "clf__kernel": ["linear"],
        "clf__C": [0.01, 0.1, 1, 3, 10, 30, 100],
        "clf__class_weight": [None, "balanced"],
    },
    {
        "clf__kernel": ["rbf"],
        "clf__C": [0.1, 1, 3, 10, 30, 100],
        "clf__gamma": ["scale", "auto", 0.01, 0.03, 0.1, 0.3, 1.0],
        "clf__class_weight": [None, "balanced"],
    },
]

gs_svm = GridSearchCV(
    estimator=pipe_svm,
    param_grid=param_grid_svm,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
    verbose=1
)
gs_svm.fit(X_train, y_train)
print("\n[SVM] Best CV macro-F1:", gs_svm.best_score_)
print("[SVM] Best params:", gs_svm.best_params_)
svm_tuned = gs_svm.best_estimator_

val_score_bad = get_score_for_auc(svm_tuned, X_val)
best_thr, best_val_f1 = tune_threshold_by_f1(y_val, val_score_bad)

test_score_bad = get_score_for_auc(svm_tuned, X_test)

val_pred_default = (val_score_bad >= 0.5).astype(int)
test_pred_default = (test_score_bad >= 0.5).astype(int)

val_pred_best = (val_score_bad >= best_thr).astype(int)
test_pred_best = (test_score_bad >= best_thr).astype(int)

val_metrics_default = metrics_dict(y_val, val_pred_default, val_score_bad, 0.5)
val_metrics_best = metrics_dict(y_val, val_pred_best, val_score_bad, best_thr)
test_metrics_default = metrics_dict(y_test, test_pred_default, test_score_bad, 0.5)
test_metrics_best = metrics_dict(y_test, test_pred_best, test_score_bad, best_thr)

print("Best threshold (SVM, bad):", best_thr, "| Val F1_bad:", best_val_f1)
print_report("SVM tuned (MFCC)", y_test, test_pred_best, test_score_bad)
add_row(rows, "SVM tuned (MFCC)", y_test, test_pred_best, test_score_bad)

log_baseline_metrics(
    run=run,
    val_default=val_metrics_default,
    val_best=val_metrics_best,
    test_default=test_metrics_default,
    test_best=test_metrics_best,
    thr_bad=best_thr,
    extra={
        "cv/best_score": float(gs_svm.best_score_),
        "model/name": "mfcc_svm",
    },
)

finish_baseline_run(run)

Fitting 5 folds for each of 98 candidates, totalling 490 fits

[SVM] Best CV macro-F1: 0.7052796033313825
[SVM] Best params: {'clf__C': 1, 'clf__class_weight': 'balanced', 'clf__gamma': 0.03, 'clf__kernel': 'rbf'}
Best threshold (SVM, bad): 0.17 | Val F1_bad: 0.5682819383259912
[SVM tuned (MFCC)]
  Accuracy      : 0.5717
  F1-bad        : 0.5647
  F1-macro      : 0.5716
  Precision_bad : 0.4212
  Recall_bad    : 0.8564
  ROC-AUC       : 0.7173
  AP            : 0.5424


cv/best_score,▁
test/best_f1_from_val_acc,▁
test/best_f1_from_val_ap,▁
test/best_f1_from_val_auc,▁
test/best_f1_from_val_f1,▁
test/best_f1_from_val_f1_macro,▁
test/best_f1_from_val_precision,▁
test/best_f1_from_val_recall,▁
test/default_acc,▁
test/default_ap,▁
+20,...


### XGBoost

In [ ]:
run = start_baseline_run(
    name="mfcc_xgboost",
    config={
        "model_family": "baseline",
        "features": "mfcc",
        "classifier": "xgboost",
        "sr": SR,
        "max_seconds": MAX_SECONDS,
        "random_state": seed,
        "n_mfcc": N_MFCC,
    },
)

pipe_xgb = Pipeline([
    ("clf", XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        random_state=seed,
        n_jobs=-1
    ))
])

param_dist_xgb = {
    "clf__n_estimators": randint(300, 1200),
    "clf__max_depth": randint(2, 10),
    "clf__learning_rate": uniform(0.01, 0.2),
    "clf__subsample": uniform(0.6, 0.4),
    "clf__colsample_bytree": uniform(0.6, 0.4),
    "clf__min_child_weight": randint(1, 10),
    "clf__gamma": uniform(0.0, 0.5)
}

rs_xgb = RandomizedSearchCV(
    estimator=pipe_xgb,
    param_distributions=param_dist_xgb,
    n_iter=40,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
    verbose=1,
    random_state=seed
)
rs_xgb.fit(X_train, y_train)
print("\n[XGB] Best CV macro-F1:", rs_xgb.best_score_)
print("[XGB] Best params:", rs_xgb.best_params_)
xgb_tuned = rs_xgb.best_estimator_

val_score_bad = get_score_for_auc(xgb_tuned, X_val)
best_thr, best_val_f1 = tune_threshold_by_f1(y_val, val_score_bad)

test_score_bad = get_score_for_auc(xgb_tuned, X_test)

val_pred_default = (val_score_bad >= 0.5).astype(int)
test_pred_default = (test_score_bad >= 0.5).astype(int)

val_pred_best = (val_score_bad >= best_thr).astype(int)
test_pred_best = (test_score_bad >= best_thr).astype(int)

val_metrics_default = metrics_dict(y_val, val_pred_default, val_score_bad, 0.5)
val_metrics_best = metrics_dict(y_val, val_pred_best, val_score_bad, best_thr)
test_metrics_default = metrics_dict(y_test, test_pred_default, test_score_bad, 0.5)
test_metrics_best = metrics_dict(y_test, test_pred_best, test_score_bad, best_thr)

print("Best threshold (XGB, bad):", best_thr, "| Val F1_bad:", best_val_f1)
print_report("XGBoost tuned (MFCC)", y_test, test_pred_best, test_score_bad)
add_row(rows, "XGBoost tuned (MFCC)", y_test, test_pred_best, test_score_bad)

log_baseline_metrics(
    run=run,
    val_default=val_metrics_default,
    val_best=val_metrics_best,
    test_default=test_metrics_default,
    test_best=test_metrics_best,
    thr_bad=best_thr,
    extra={
        "cv/best_score": float(rs_xgb.best_score_),
        "model/name": "mfcc_xgboost",
    },
)

finish_baseline_run(run)

Fitting 5 folds for each of 40 candidates, totalling 200 fits

[XGB] Best CV macro-F1: 0.6941615231317497
[XGB] Best params: {'clf__colsample_bytree': np.float64(0.6592347719813599), 'clf__gamma': np.float64(0.49887024252447093), 'clf__learning_rate': np.float64(0.063356202855057), 'clf__max_depth': 3, 'clf__min_child_weight': 2, 'clf__n_estimators': 519, 'clf__subsample': np.float64(0.6205915004999957)}
Best threshold (XGB, bad): 0.18000000000000002 | Val F1_bad: 0.5960591133004927
[XGBoost tuned (MFCC)]
  Accuracy      : 0.5771
  F1-bad        : 0.5280
  F1-macro      : 0.5724
  Precision_bad : 0.4138
  Recall_bad    : 0.7293
  ROC-AUC       : 0.7116
  AP            : 0.5781


cv/best_score,▁
test/best_f1_from_val_acc,▁
test/best_f1_from_val_ap,▁
test/best_f1_from_val_auc,▁
test/best_f1_from_val_f1,▁
test/best_f1_from_val_f1_macro,▁
test/best_f1_from_val_precision,▁
test/best_f1_from_val_recall,▁
test/default_acc,▁
test/default_ap,▁
+20,...


In [ ]:
tab_results = pd.DataFrame(rows).sort_values("F1_bad", ascending=False).reset_index(drop=True)
display(tab_results)

,Model,Accuracy,F1_bad,F1_macro,Precision_bad,Recall_bad,Recall_good,ROC_AUC,AP
0,SVM tuned (MFCC),0.677419,0.565217,0.654404,0.502146,0.646409,0.692308,0.735107,0.595436
1,LogReg tuned (MFCC),0.607527,0.557576,0.602459,0.439490,0.762431,0.533156,0.708780,0.530343
2,XGBoost tuned (MFCC),0.641577,0.547511,0.625388,0.463602,0.668508,0.628647,0.703665,0.548935


## CNN на Mel-спектрограммах

In [ ]:
df_train_cnn = df_train_base.copy().reset_index(drop=True)
df_val_cnn   = df_val_base.copy().reset_index(drop=True)
df_test_cnn  = df_test_base.copy().reset_index(drop=True)

print("Train:", len(df_train_cnn), "Val:", len(df_val_cnn), "Test:", len(df_test_cnn))
print("Train counts:\n", df_train_cnn["label"].value_counts())
print("Val counts:\n", df_val_cnn["label"].value_counts())
print("Test counts:\n", df_test_cnn["label"].value_counts())

Train: 1785 Val: 447 Test: 558
Train counts:
 label
0    1206
1     579
Name: count, dtype: int64
Val counts:
 label
0    302
1    145
Name: count, dtype: int64
Test counts:
 label
0    377
1    181
Name: count, dtype: int64


In [ ]:
N_MELS = 64
TARGET_T = 256

def mel_spec(x, sr=SR, n_mels=N_MELS, hop_length=256, n_fft=1024):
    S = librosa.feature.melspectrogram(y=x, sr=sr, n_mels=n_mels, hop_length=hop_length, n_fft=n_fft, power=2.0)
    S = librosa.power_to_db(S, ref=np.max)
    return S.astype(np.float32)

def fix_time_dim(S, target_T=TARGET_T):
    T = S.shape[1]
    if T > target_T:
        start = (T - target_T) // 2
        return S[:, start:start + target_T]
    if T < target_T:
        pad = target_T - T
        return np.pad(S, ((0, 0), (0, pad)), mode="constant")
    return S

def delta_feats(S):
    d1 = librosa.feature.delta(S)
    d2 = librosa.feature.delta(S, order=2)
    return d1.astype(np.float32), d2.astype(np.float32)

class SpeechSpecDataset(Dataset):
    def __init__(self, df_part: pd.DataFrame, train=True):
        self.paths = df_part["path"].tolist()
        self.labels = df_part["label"].astype(int).tolist()
        self.train = train

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        path = self.paths[idx]
        y = int(self.labels[idx])

        x = load_audio(path, sr=SR)
        x = pad_or_crop(x, max_len=MAX_LEN, train=self.train)

        S = mel_spec(x, sr=SR, n_mels=N_MELS)
        S = fix_time_dim(S, target_T=TARGET_T)

        d1, d2 = delta_feats(S)
        X = np.stack([S, d1, d2], axis=0)

        return torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.long)

In [ ]:
class ImprovedCNN(nn.Module):
    def __init__(self, in_ch=3, n_classes=2, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Dropout(dropout))
        with torch.no_grad():
            dummy = torch.zeros(1, in_ch, N_MELS, TARGET_T)
            h = self.net(dummy).view(1, -1).shape[1]
        self.head = nn.Sequential(
            nn.Linear(h, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, n_classes))

    def forward(self, x):
        x = self.net(x)
        x = x.view(x.size(0), -1)
        return self.head(x)

In [ ]:
def make_class_weights_from_df(df_part, device):
    counts = df_part["label"].value_counts().sort_index()
    n0, n1 = counts.loc[0], counts.loc[1]
    total = n0 + n1
    w0 = total / (2 * n0)
    w1 = total / (2 * n1)
    return torch.tensor([w0, w1], dtype=torch.float32, device=device)

def run_epoch_cnn(model, loader, criterion, device, optimizer=None, train=True):
    if train:
        model.train()
        if optimizer is None:
            raise ValueError("optimizer отсутствует")
    else:
        model.eval()

    losses, ys, preds, probs_bad = [], [], [], []

    with torch.set_grad_enabled(train):
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            losses.append(loss.item())

            prob_bad = torch.softmax(logits, dim=1)[:, 1].detach().cpu().numpy()
            pred_labels = (prob_bad >= 0.5).astype(int)

            probs_bad.append(prob_bad)
            ys.extend(yb.detach().cpu().numpy())
            preds.extend(pred_labels)

    probs_bad = np.concatenate(probs_bad) if len(probs_bad) else np.array([])
    ys = np.array(ys)
    preds = np.array(preds)

    f1_bad = f1_score(ys, preds, average="binary", pos_label=1)
    return float(np.mean(losses)), float(f1_bad), ys, preds, probs_bad

In [ ]:
train_ds = SpeechSpecDataset(df_train_cnn, train=True)
val_ds   = SpeechSpecDataset(df_val_cnn, train=False)
test_ds  = SpeechSpecDataset(df_test_cnn, train=False)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=2)

model_cnn = ImprovedCNN(in_ch=3, dropout=0.3).to(device)

class_weights_cnn = make_class_weights_from_df(df_train_cnn, device)
criterion_cnn = nn.CrossEntropyLoss(weight=class_weights_cnn)

optimizer_cnn = torch.optim.Adam(model_cnn.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler_cnn = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_cnn, mode="max", factor=0.5, patience=2
)

best_val_f1 = -1.0
best_state = None
best_epoch = -1

run = start_baseline_run(
    name="mel_cnn",
    config={
        "model_family": "baseline",
        "features": "mel+delta+delta2",
        "classifier": "cnn",
        "sr": SR,
        "max_seconds": MAX_SECONDS,
        "random_state": seed,
        "epochs": 12,
        "batch_size": 32,
        "n_mels": N_MELS,
        "target_t": TARGET_T,
    },
)

In [ ]:
EPOCHS = 12

for epoch in tqdm(range(1, EPOCHS + 1)):
    tr_loss, tr_f1, *_ = run_epoch_cnn(
        model_cnn, train_loader, criterion_cnn, device,
        optimizer=optimizer_cnn, train=True
    )
    va_loss, va_f1, va_y, va_pred, va_prob_bad = run_epoch_cnn(
        model_cnn, val_loader, criterion_cnn, device,
        optimizer=None, train=False
    )

    print(f"Epoch {epoch:02d} | train loss {tr_loss:.4f} f1_bad {tr_f1:.3f} | val loss {va_loss:.4f} f1_bad {va_f1:.3f}")
    scheduler_cnn.step(va_f1)

    if run is not None:
        wandb.log({
            "epoch": epoch,
            "train/loss": float(tr_loss),
            "train/f1_bad": float(tr_f1),
            "val/loss": float(va_loss),
            "val/f1_bad": float(va_f1),
            "lr": float(optimizer_cnn.param_groups[0]["lr"]),
        })

    if va_f1 > best_val_f1:
        best_val_f1 = va_f1
        best_epoch = epoch
        best_state = {k: v.detach().cpu().clone() for k, v in model_cnn.state_dict().items()}

print("Best val F1_bad:", best_val_f1, "| Best epoch:", best_epoch)

model_cnn.load_state_dict(best_state)
model_cnn.to(device)

va_loss, va_f1, va_y, va_pred_default, va_prob_bad = run_epoch_cnn(
    model_cnn, val_loader, criterion_cnn, device,
    optimizer=None, train=False
)
best_thr_cnn, best_val_f1_cnn = tune_threshold_by_f1(va_y, va_prob_bad)

te_loss, te_f1, te_y, te_pred_default, te_prob_bad = run_epoch_cnn(
    model_cnn, test_loader, criterion_cnn, device,
    optimizer=None, train=False
)

val_pred_default = va_pred_default
test_pred_default = te_pred_default

val_pred_best = (va_prob_bad >= best_thr_cnn).astype(int)

test_pred_best = (te_prob_bad >= best_thr_cnn).astype(int)

val_metrics_default = metrics_dict(va_y, val_pred_default, va_prob_bad, 0.5)
val_metrics_best = metrics_dict(va_y, val_pred_best, va_prob_bad, best_thr_cnn)
test_metrics_default = metrics_dict(te_y, test_pred_default, te_prob_bad, 0.5)
test_metrics_best = metrics_dict(te_y, test_pred_best, te_prob_bad, best_thr_cnn)

print("Best threshold (CNN, bad):", best_thr_cnn, "| Val F1_bad:", best_val_f1_cnn)
print_report("CNN (Mel+delta+delta2)", te_y, test_pred_best, te_prob_bad)
add_row(rows, "CNN (Mel+delta+delta2)", te_y, test_pred_best, te_prob_bad)

log_baseline_metrics(
    run=run,
    val_default=val_metrics_default,
    val_best=val_metrics_best,
    test_default=test_metrics_default,
    test_best=test_metrics_best,
    thr_bad=best_thr_cnn,
    extra={
        "best/epoch": int(best_epoch),
        "best/val_f1_bad": float(best_val_f1),
        "model/name": "mel_cnn",
    },
)

finish_baseline_run(run)

  8%|▊         | 1/12 [00:45<08:18, 45.28s/it]

Epoch 01 | train loss 1.1479 f1_bad 0.447 | val loss 0.6564 f1_bad 0.481


 17%|█▋        | 2/12 [01:31<07:40, 46.05s/it]

Epoch 02 | train loss 0.6403 f1_bad 0.494 | val loss 0.6456 f1_bad 0.464


 25%|██▌       | 3/12 [02:16<06:49, 45.46s/it]

...

Epoch 11 | train loss 0.5668 f1_bad 0.587 | val loss 0.6523 f1_bad 0.523


100%|██████████| 12/12 [08:51<00:00, 44.31s/it]

Epoch 12 | train loss 0.5561 f1_bad 0.604 | val loss 0.6261 f1_bad 0.525
Best val F1_bad: 0.54421768707483 | Best epoch: 5


Best threshold (CNN, bad): 0.44 | Val F1_bad: 0.5546218487394958
[CNN (Mel+delta+delta2)]
  Accuracy      : 0.5251
  F1-bad        : 0.5501
  F1-macro      : 0.5236
  Precision_bad : 0.3971
  Recall_bad    : 0.8950
  ROC-AUC       : 0.6731
  AP            : 0.5168


best/epoch,▁
best/val_f1_bad,▁
epoch,▁▂▂▃▄▄▅▅▆▇▇█
lr,███████▃▃▃▁▁
test/best_f1_from_val_acc,▁
test/best_f1_from_val_ap,▁
test/best_f1_from_val_auc,▁
test/best_f1_from_val_f1,▁
test/best_f1_from_val_f1_macro,▁
test/best_f1_from_val_precision,▁
+27,...


In [ ]:
all_results = pd.DataFrame(rows).sort_values("F1_bad", ascending=False).reset_index(drop=True)
display(all_results)

,Model,Accuracy,F1_bad,F1_macro,Precision_bad,Recall_bad,Recall_good,ROC_AUC,AP
0,CNN (Mel+delta+delta2),0.668459,0.586130,0.654799,0.492481,0.723757,0.641910,0.721962,0.570597
1,SVM tuned (MFCC),0.677419,0.565217,0.654404,0.502146,0.646409,0.692308,0.735107,0.595436
2,LogReg tuned (MFCC),0.607527,0.557576,0.602459,0.439490,0.762431,0.533156,0.708780,0.530343
3,XGBoost tuned (MFCC),0.641577,0.547511,0.625388,0.463602,0.668508,0.628647,0.703665,0.548935


## Модели на WavLM-эмбеддингах

In [ ]:
from transformers import AutoFeatureExtractor, WavLMModel

MODEL_NAME = "microsoft/wavlm-base"
SR = 16000
MAX_SECONDS = 10.0
MAX_LEN = int(SR * MAX_SECONDS)

device = "cuda" if torch.cuda.is_available() else "cpu"

processor = AutoFeatureExtractor.from_pretrained(MODEL_NAME)
WavLM = WavLMModel.from_pretrained(MODEL_NAME).to(device)
WavLM.eval()
for p in WavLM.parameters():
    p.requires_grad = False

@torch.no_grad()
def wavlm_embed_batch(paths, batch_size=8, max_len=MAX_LEN, train=False):
    embs = []

    for i in range(0, len(paths), batch_size):
        batch_paths = paths[i:i + batch_size]

        audios = []
        for p in batch_paths:
            x = load_audio(p, sr=SR)
            x = pad_or_crop(x, max_len=max_len, train=train)
            audios.append(x.astype(np.float32))

        inputs = processor(
            audios,
            sampling_rate=SR,
            return_tensors="pt",
            padding=True
        )

        input_values = inputs["input_values"].to(device)
        attention_mask = inputs.get("attention_mask")
        if attention_mask is not None:
            attention_mask = attention_mask.to(device)

        out = WavLM(input_values=input_values, attention_mask=attention_mask)
        h = out.last_hidden_state

        pooled = h.mean(dim=1)

        embs.append(pooled.cpu().numpy())

    return np.concatenate(embs, axis=0)

X_train_WavLM = wavlm_embed_batch(df_train_base["path"].tolist(), batch_size=8, train=True)
X_val_WavLM   = wavlm_embed_batch(df_val_base["path"].tolist(),   batch_size=8, train=False)
X_test_WavLM  = wavlm_embed_batch(df_test_base["path"].tolist(),  batch_size=8, train=False)

y_train_WavLM = df_train_base["label"].astype(int).values
y_val_WavLM   = df_val_base["label"].astype(int).values
y_test_WavLM  = df_test_base["label"].astype(int).values

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/378M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/248 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(


### XGBoost (WavLM)

In [ ]:
run = start_baseline_run(
    name="wavlm_emb_xgboost",
    config={
        "model_family": "baseline",
        "features": "wavlm_embedding",
        "classifier": "xgboost",
        "encoder_name": MODEL_NAME,
        "sr": SR,
        "max_seconds": MAX_SECONDS,
        "random_state": seed,
    },
)

xgb = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    random_state=seed,
    n_jobs=-1
)

param_dist_xgb = {
    "n_estimators": randint(300, 1500),
    "max_depth": randint(2, 10),
    "learning_rate": uniform(0.01, 0.2),
    "subsample": uniform(0.6, 0.4),
    "colsample_bytree": uniform(0.6, 0.4),
    "min_child_weight": randint(1, 10),
    "gamma": uniform(0.0, 0.5),
    "reg_alpha": uniform(0.0, 1.0),
    "reg_lambda": uniform(0.5, 2.0),
}

rs_xgb = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_dist_xgb,
    n_iter=40,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
    verbose=1,
    random_state=seed
)

rs_xgb.fit(X_train_WavLM, y_train_WavLM)
print("\n[XGB on WavLM emb] Best CV macro-F1:", rs_xgb.best_score_)
print("[XGB on WavLM emb] Best params:", rs_xgb.best_params_)

xgb_WavLM = rs_xgb.best_estimator_

val_score_bad = xgb_WavLM.predict_proba(X_val_WavLM)[:, 1]
best_thr, best_val_f1 = tune_threshold_by_f1(y_val_WavLM, val_score_bad)

test_score_bad = xgb_WavLM.predict_proba(X_test_WavLM)[:, 1]

val_pred_default = (val_score_bad >= 0.5).astype(int)
test_pred_default = (test_score_bad >= 0.5).astype(int)

val_pred_best = (val_score_bad >= best_thr).astype(int)
test_pred_best = (test_score_bad >= best_thr).astype(int)

val_metrics_default = metrics_dict(y_val_WavLM, val_pred_default, val_score_bad, 0.5)
val_metrics_best = metrics_dict(y_val_WavLM, val_pred_best, val_score_bad, best_thr)
test_metrics_default = metrics_dict(y_test_WavLM, test_pred_default, test_score_bad, 0.5)
test_metrics_best = metrics_dict(y_test_WavLM, test_pred_best, test_score_bad, best_thr)

print("Best threshold (XGB WavLM, bad):", best_thr, "| Val F1_bad:", best_val_f1)
print_report("XGBoost (WavLM emb)", y_test_WavLM, test_pred_best, test_score_bad)
add_row(rows, "XGBoost (WavLM emb)", y_test_WavLM, test_pred_best, test_score_bad)

log_baseline_metrics(
    run=run,
    val_default=val_metrics_default,
    val_best=val_metrics_best,
    test_default=test_metrics_default,
    test_best=test_metrics_best,
    thr_bad=best_thr,
    extra={
        "cv/best_score": float(rs_xgb.best_score_),
        "model/name": "wavlm_emb_xgboost",
        "encoder_name": MODEL_NAME,
    },
)

finish_baseline_run(run)

pd.DataFrame(rows).sort_values("F1_bad", ascending=False)

Fitting 5 folds for each of 40 candidates, totalling 200 fits

[XGB on WavLM emb] Best CV macro-F1: 0.771497905553244
[XGB on WavLM emb] Best params: {'colsample_bytree': np.float64(0.803525630735504), 'gamma': np.float64(0.3181663090929477), 'learning_rate': np.float64(0.060092363721116836), 'max_depth': 7, 'min_child_weight': 6, 'n_estimators': 529, 'reg_alpha': np.float64(0.3930977246667604), 'reg_lambda': np.float64(2.2840931103542266), 'subsample': np.float64(0.8524554503989051)}
Best threshold (XGB WavLM, bad): 0.25 | Val F1_bad: 0.6840390879478827
[XGBoost (WavLM emb)]
  Accuracy      : 0.7706
  F1-bad        : 0.6425
  F1-macro      : 0.7368
  Precision_bad : 0.6497
  Recall_bad    : 0.6354
  ROC-AUC       : 0.8014
  AP            : 0.7046


cv/best_score,▁
test/best_f1_from_val_acc,▁
test/best_f1_from_val_ap,▁
test/best_f1_from_val_auc,▁
test/best_f1_from_val_f1,▁
test/best_f1_from_val_f1_macro,▁
test/best_f1_from_val_precision,▁
test/best_f1_from_val_recall,▁
test/default_acc,▁
test/default_ap,▁
+20,...


,Model,Accuracy,F1_bad,F1_macro,Precision_bad,Recall_bad,Recall_good,ROC_AUC,AP
5,XGBoost (WavLM emb),0.770609,0.642458,0.736796,0.649718,0.635359,0.835544,0.801442,0.704642
2,SVM tuned (MFCC),0.571685,0.564663,0.571573,0.421196,0.856354,0.435013,0.717294,0.542405
4,CNN (Mel+delta+delta2),0.525090,0.550085,0.523619,0.397059,0.895028,0.347480,0.673110,0.516838
3,XGBoost tuned (MFCC),0.577061,0.528000,0.572442,0.413793,0.729282,0.503979,0.711623,0.578090
1,LogReg tuned (MFCC),0.650538,0.516129,0.621318,0.468468,0.574586,0.687003,0.671996,0.511290
0,LogReg tuned (MFCC),0.650538,0.516129,0.621318,0.468468,0.574586,0.687003,0.671996,0.511290


## SVM (WavLM)

In [ ]:
run = start_baseline_run(
    name="wavlm_emb_svm",
    config={
        "model_family": "baseline",
        "features": "wavlm_embedding",
        "classifier": "svm",
        "encoder_name": MODEL_NAME,
        "sr": SR,
        "max_seconds": MAX_SECONDS,
        "random_state": seed,
    },
)

pipe_svm = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", SVC(probability=True))
])

param_grid_svm = [
    {"clf__kernel":["linear"], "clf__C":[0.1, 1, 3, 10, 30],
     "clf__class_weight":[None, "balanced"]},
    {"clf__kernel":["rbf"], "clf__C":[0.1, 1, 3, 10, 30],
     "clf__gamma":["scale", "auto", 0.01, 0.03, 0.1],
     "clf__class_weight":[None, "balanced"]}
]

gs_svm = GridSearchCV(
    pipe_svm,
    param_grid_svm,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
    verbose=1
)
gs_svm.fit(X_train_WavLM, y_train_WavLM)

print("\n[SVM on WavLM emb] Best CV macro-F1:", gs_svm.best_score_)
print("[SVM on WavLM emb] Best params:", gs_svm.best_params_)

svm_WavLM = gs_svm.best_estimator_

val_score_bad = get_score_for_auc(svm_WavLM, X_val_WavLM)
best_thr, best_val_f1 = tune_threshold_by_f1(y_val_WavLM, val_score_bad)

test_score_bad = get_score_for_auc(svm_WavLM, X_test_WavLM)

val_pred_default = (val_score_bad >= 0.5).astype(int)
test_pred_default = (test_score_bad >= 0.5).astype(int)

val_pred_best = (val_score_bad >= best_thr).astype(int)
test_pred_best = (test_score_bad >= best_thr).astype(int)

val_metrics_default = metrics_dict(y_val_WavLM, val_pred_default, val_score_bad, 0.5)
val_metrics_best = metrics_dict(y_val_WavLM, val_pred_best, val_score_bad, best_thr)
test_metrics_default = metrics_dict(y_test_WavLM, test_pred_default, test_score_bad, 0.5)
test_metrics_best = metrics_dict(y_test_WavLM, test_pred_best, test_score_bad, best_thr)

print("Best threshold (SVM WavLM, bad):", best_thr, "| Val F1_bad:", best_val_f1)
print_report("SVM (WavLM emb)", y_test_WavLM, test_pred_best, test_score_bad)
add_row(rows, "SVM (WavLM emb)", y_test_WavLM, test_pred_best, test_score_bad)

log_baseline_metrics(
    run=run,
    val_default=val_metrics_default,
    val_best=val_metrics_best,
    test_default=test_metrics_default,
    test_best=test_metrics_best,
    thr_bad=best_thr,
    extra={
        "cv/best_score": float(gs_svm.best_score_),
        "model/name": "wavlm_emb_svm",
        "encoder_name": MODEL_NAME,
    },
)

finish_baseline_run(run)

pd.DataFrame(rows).sort_values("F1_bad", ascending=False)

Fitting 5 folds for each of 60 candidates, totalling 300 fits

[SVM on WavLM emb] Best CV macro-F1: 0.7744029715997482
[SVM on WavLM emb] Best params: {'clf__C': 1, 'clf__class_weight': 'balanced', 'clf__gamma': 'scale', 'clf__kernel': 'rbf'}
Best threshold (SVM WavLM, bad): 0.31 | Val F1_bad: 0.668769716088328
[SVM (WavLM emb)]
  Accuracy      : 0.7545
  F1-bad        : 0.6442
  F1-macro      : 0.7284
  Precision_bad : 0.6078
  Recall_bad    : 0.6851
  ROC-AUC       : 0.8050
  AP            : 0.6914


cv/best_score,▁
test/best_f1_from_val_acc,▁
test/best_f1_from_val_ap,▁
test/best_f1_from_val_auc,▁
test/best_f1_from_val_f1,▁
test/best_f1_from_val_f1_macro,▁
test/best_f1_from_val_precision,▁
test/best_f1_from_val_recall,▁
test/default_acc,▁
test/default_ap,▁
+20,...


,Model,Accuracy,F1_bad,F1_macro,Precision_bad,Recall_bad,Recall_good,ROC_AUC,AP
6,SVM (WavLM emb),0.754480,0.644156,0.728371,0.607843,0.685083,0.787798,0.804988,0.691444
5,XGBoost (WavLM emb),0.770609,0.642458,0.736796,0.649718,0.635359,0.835544,0.801442,0.704642
2,SVM tuned (MFCC),0.571685,0.564663,0.571573,0.421196,0.856354,0.435013,0.717294,0.542405
4,CNN (Mel+delta+delta2),0.525090,0.550085,0.523619,0.397059,0.895028,0.347480,0.673110,0.516838
3,XGBoost tuned (MFCC),0.577061,0.528000,0.572442,0.413793,0.729282,0.503979,0.711623,0.578090
1,LogReg tuned (MFCC),0.650538,0.516129,0.621318,0.468468,0.574586,0.687003,0.671996,0.511290
0,LogReg tuned (MFCC),0.650538,0.516129,0.621318,0.468468,0.574586,0.687003,0.671996,0.511290


### WavLM бинарная классификация

In [ ]:
from transformers import AutoModel
from copy import deepcopy
import torchaudio

In [ ]:
class RawAudioDataset(Dataset):
    def __init__(self, df, sr=16000, max_seconds=10.0):
        self.df = df.reset_index(drop=True)
        self.sr = sr
        self.max_len = int(sr * max_seconds)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        path = self.df.loc[idx, "path"]
        label = int(self.df.loc[idx, "label"])

        wav, orig_sr = torchaudio.load(path)
        if wav.shape[0] > 1:
            wav = wav.mean(dim=0, keepdim=True)

        if orig_sr != self.sr:
            wav = torchaudio.functional.resample(wav, orig_sr, self.sr)

        wav = wav.squeeze(0)

        if wav.numel() < self.max_len:
            pad = self.max_len - wav.numel()
            wav = torch.nn.functional.pad(wav, (0, pad))
        else:
            wav = wav[:self.max_len]

        attn_mask = (wav.abs() > 1e-8).long()

        return {
            "audio": wav,
            "attention_mask": attn_mask,
            "label": torch.tensor(label, dtype=torch.long),
        }

In [ ]:
class WavLMBinaryClassifier(nn.Module):
    def __init__(self, encoder_name="microsoft/wavlm-base", dropout=0.2, freeze_encoder=True):
        super().__init__()
        self.encoder_name = encoder_name
        self.encoder = AutoModel.from_pretrained(encoder_name)

        if freeze_encoder:
            for p in self.encoder.parameters():
                p.requires_grad = False

        hidden_size = self.encoder.config.hidden_size

        self.head = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 2)
        )

    def forward(self, audio, attention_mask=None):
        out = self.encoder(input_values=audio, attention_mask=attention_mask)
        h = out.last_hidden_state

        if attention_mask is not None and hasattr(self.encoder, "_get_feature_vector_attention_mask"):
            feat_mask = self.encoder._get_feature_vector_attention_mask(
                h.shape[1], attention_mask
            )
            feat_mask = feat_mask.unsqueeze(-1).float()
            h = h * feat_mask
            emb = h.sum(dim=1) / feat_mask.sum(dim=1).clamp(min=1e-6)
        else:
            emb = h.mean(dim=1)

        logits = self.head(emb)
        return logits

In [ ]:
def unfreeze_last_n_layers(model, n_last=4):
    for p in model.encoder.parameters():
        p.requires_grad = False

    if n_last <= 0:
        return

    if hasattr(model.encoder, "encoder") and hasattr(model.encoder.encoder, "layers"):
        layers = model.encoder.encoder.layers
    elif hasattr(model.encoder, "layers"):
        layers = model.encoder.layers
    else:
        raise ValueError("Cannot find encoder layers")

    for layer in layers[-n_last:]:
        for p in layer.parameters():
            p.requires_grad = True

In [ ]:
print([name for name in globals().keys() if "train" in name.lower() or "val" in name.lower() or "test" in name.lower()])

['train_test_split', 'df_train_tab', 'df_test_tab', 'df_train_base', 'df_val_base', 'df_test_base', 'X_train', 'y_train', 'X_val', 'y_val', 'X_test', 'y_test', 'val_score_bad', 'best_val_f1', 'test_score_bad', 'val_pred_default', 'test_pred_default', 'val_pred_best', 'test_pred_best', 'val_metrics_default', 'val_metrics_best', 'test_metrics_default', 'test_metrics_best', 'df_train_cnn', 'df_val_cnn', 'df_test_cnn', 'train_ds', 'val_ds', 'test_ds', 'train_loader', 'val_loader', 'test_loader', 'best_val_f1_cnn', 'X_train_WavLM', 'X_val_WavLM', 'X_test_WavLM', 'y_train_WavLM', 'y_val_WavLM', 'y_test_WavLM']


In [ ]:
wavlm_bin_train_ds = RawAudioDataset(df_train_base, sr=SR, max_seconds=MAX_SECONDS)
wavlm_bin_val_ds   = RawAudioDataset(df_val_base, sr=SR, max_seconds=MAX_SECONDS)
wavlm_bin_test_ds  = RawAudioDataset(df_test_base, sr=SR, max_seconds=MAX_SECONDS)

wavlm_bin_train_loader = DataLoader(wavlm_bin_train_ds, batch_size=8, shuffle=True, num_workers=2)
wavlm_bin_val_loader   = DataLoader(wavlm_bin_val_ds, batch_size=8, shuffle=False, num_workers=2)
wavlm_bin_test_loader  = DataLoader(wavlm_bin_test_ds, batch_size=8, shuffle=False, num_workers=2)

In [ ]:
def run_epoch_binary(model, loader, criterion, device, optimizer=None, train=False):
    model.train(train)

    all_y = []
    all_pred = []
    all_prob_bad = []
    total_loss = 0.0
    n = 0

    for batch in loader:
        audio = batch["audio"].to(device)
        attn = batch["attention_mask"].to(device)
        y = batch["label"].to(device)

        with torch.set_grad_enabled(train):
            logits = model(audio, attention_mask=attn)
            loss = criterion(logits, y)

            if train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

        prob_bad = torch.softmax(logits, dim=1)[:, 1].detach().cpu().numpy()
        pred = (prob_bad >= 0.5).astype(int)

        all_y.append(y.detach().cpu().numpy())
        all_pred.append(pred)
        all_prob_bad.append(prob_bad)

        bs = y.size(0)
        total_loss += loss.item() * bs
        n += bs

    y_true = np.concatenate(all_y)
    y_pred = np.concatenate(all_pred)
    y_prob_bad = np.concatenate(all_prob_bad)

    avg_loss = total_loss / max(n, 1)
    f1_bad = f1_score(y_true, y_pred, average="binary", pos_label=1)

    return avg_loss, f1_bad, y_true, y_pred, y_prob_bad

In [ ]:
wavlm_bin_run = start_baseline_run(
    name="wavlm_binary",
    config={
        "model_family": "baseline",
        "features": "raw_audio",
        "classifier": "wavlm_binary_head",
        "encoder_name": "microsoft/wavlm-base",
        "sr": SR,
        "max_seconds": MAX_SECONDS,
        "random_state": seed,
        "epochs": 12,
        "batch_size": 8,
        "freeze_mode": "unfreeze_last_n",
        "unfreeze_last_n": 4,
    },
)

wavlm_bin_model = WavLMBinaryClassifier(
    encoder_name="microsoft/wavlm-base",
    dropout=0.2,
    freeze_encoder=True,
).to(device)

unfreeze_last_n_layers(wavlm_bin_model, n_last=4)

criterion_bin = nn.CrossEntropyLoss()

head_params = list(wavlm_bin_model.head.parameters())
enc_params = [p for p in wavlm_bin_model.encoder.parameters() if p.requires_grad]

optimizer_bin = torch.optim.AdamW(
    [
        {"params": head_params, "lr": 3e-4},
        {"params": enc_params, "lr": 1e-5},
    ],
    weight_decay=1e-2,
)

scheduler_bin = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_bin,
    mode="min",
    factor=0.5,
    patience=1,
)

best_val_loss = float("inf")
best_epoch = -1
best_state = None

EPOCHS = 12

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_f1, *_ = run_epoch_binary(
        wavlm_bin_model,
        wavlm_bin_train_loader,
        criterion_bin,
        device,
        optimizer=optimizer_bin,
        train=True,
    )

    va_loss, va_f1, va_y, va_pred_default, va_prob_bad = run_epoch_binary(
        wavlm_bin_model,
        wavlm_bin_val_loader,
        criterion_bin,
        device,
        optimizer=None,
        train=False,
    )

    scheduler_bin.step(va_loss)

    print(f"Epoch {epoch:02d} | train loss {tr_loss:.4f} f1_bad {tr_f1:.3f} | val loss {va_loss:.4f} f1_bad {va_f1:.3f}")

    if wavlm_bin_run is not None:
        wandb.log({
            "epoch": epoch,
            "train/loss": float(tr_loss),
            "train/f1_bad": float(tr_f1),
            "val/loss": float(va_loss),
            "val/f1_bad": float(va_f1),
            "lr/head": float(optimizer_bin.param_groups[0]["lr"]),
            "lr/encoder": float(optimizer_bin.param_groups[1]["lr"]),
        })

    if va_loss < best_val_loss:
        best_val_loss = va_loss
        best_epoch = epoch
        best_state = deepcopy(wavlm_bin_model.state_dict())

print("Best epoch:", best_epoch, "| Best val loss:", best_val_loss)

Loading weights:   0%|          | 0/248 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(


Epoch 01 | train loss 0.5115 f1_bad 0.539 | val loss 0.5204 f1_bad 0.576


/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(


Epoch 02 | train loss 0.4446 f1_bad 0.668 | val loss 0.4721 f1_bad 0.632


...

Epoch 10 | train loss 0.2755 f1_bad 0.845 | val loss 0.6226 f1_bad 0.675


/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(


Epoch 11 | train loss 0.2850 f1_bad 0.840 | val loss 0.5868 f1_bad 0.686


/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(


Epoch 12 | train loss 0.2693 f1_bad 0.843 | val loss 0.6037 f1_bad 0.697
Best epoch: 3 | Best val loss: 0.4537377766848144


In [ ]:
wavlm_bin_model.load_state_dict(best_state)

va_loss, va_f1, va_y, va_pred_default, va_prob_bad = run_epoch_binary(
    wavlm_bin_model,
    wavlm_bin_val_loader,
    criterion_bin,
    device,
    optimizer=None,
    train=False,
)

best_thr_wavlm_bin, best_val_f1_wavlm_bin = tune_threshold_by_f1(va_y, va_prob_bad)

te_loss, te_f1, te_y, te_pred_default, te_prob_bad = run_epoch_binary(
    wavlm_bin_model,
    wavlm_bin_test_loader,
    criterion_bin,
    device,
    optimizer=None,
    train=False,
)

val_pred_default = (va_prob_bad >= 0.5).astype(int)
test_pred_default = (te_prob_bad >= 0.5).astype(int)

val_pred_best = (va_prob_bad >= best_thr_wavlm_bin).astype(int)
test_pred_best = (te_prob_bad >= best_thr_wavlm_bin).astype(int)

val_metrics_default = metrics_dict(va_y, val_pred_default, va_prob_bad, 0.5)
val_metrics_best = metrics_dict(va_y, val_pred_best, va_prob_bad, best_thr_wavlm_bin)

test_metrics_default = metrics_dict(te_y, test_pred_default, te_prob_bad, 0.5)
test_metrics_best = metrics_dict(te_y, test_pred_best, te_prob_bad, best_thr_wavlm_bin)

print("Best threshold (WavLM binary):", best_thr_wavlm_bin, "| Val F1_bad:", best_val_f1_wavlm_bin)
print_report("WavLM binary classifier", te_y, test_pred_best, te_prob_bad)
add_row(rows, "WavLM binary classifier", te_y, test_pred_best, te_prob_bad)

log_baseline_metrics(
    run=wavlm_bin_run,
    val_default=val_metrics_default,
    val_best=val_metrics_best,
    test_default=test_metrics_default,
    test_best=test_metrics_best,
    thr_bad=best_thr_wavlm_bin,
    extra={
        "best/epoch": int(best_epoch),
        "best/val_loss": float(best_val_loss),
        "best/val_f1_bad": float(best_val_f1_wavlm_bin),
        "model/name": "wavlm_binary_classifier",
        "encoder_name": "microsoft/wavlm-base",
    },
)

finish_baseline_run(wavlm_bin_run)

pd.DataFrame(rows).sort_values("F1_bad", ascending=False)

/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(


Best threshold (WavLM binary): 0.6 | Val F1_bad: 0.696969696969697
[WavLM binary classifier]
  Accuracy      : 0.7563
  F1-bad        : 0.5802
  F1-macro      : 0.7043
  Precision_bad : 0.6573
  Recall_bad    : 0.5193
  ROC-AUC       : 0.8162
  AP            : 0.7178


best/epoch,▁
best/val_f1_bad,▁
best/val_loss,▁
epoch,▁▂▂▃▄▄▅▅▆▇▇█
lr/encoder,████▄▄▂▂▁▁▁▁
lr/head,████▄▄▂▂▁▁▁▁
test/best_f1_from_val_acc,▁
test/best_f1_from_val_ap,▁
test/best_f1_from_val_auc,▁
test/best_f1_from_val_f1,▁
+29,...


,Model,Accuracy,F1_bad,F1_macro,Precision_bad,Recall_bad,Recall_good,ROC_AUC,AP
6,SVM (WavLM emb),0.754480,0.644156,0.728371,0.607843,0.685083,0.787798,0.804988,0.691444
5,XGBoost (WavLM emb),0.770609,0.642458,0.736796,0.649718,0.635359,0.835544,0.801442,0.704642
7,WavLM binary classifier,0.756272,0.580247,0.704265,0.657343,0.519337,0.870027,0.816243,0.717810
2,SVM tuned (MFCC),0.571685,0.564663,0.571573,0.421196,0.856354,0.435013,0.717294,0.542405
4,CNN (Mel+delta+delta2),0.525090,0.550085,0.523619,0.397059,0.895028,0.347480,0.673110,0.516838
3,XGBoost tuned (MFCC),0.577061,0.528000,0.572442,0.413793,0.729282,0.503979,0.711623,0.578090
0,LogReg tuned (MFCC),0.650538,0.516129,0.621318,0.468468,0.574586,0.687003,0.671996,0.511290
1,LogReg tuned (MFCC),0.650538,0.516129,0.621318,0.468468,0.574586,0.687003,0.671996,0.511290
